In [2]:
!pip install xlsxwriter -q

import pandas as pd
import numpy as np
import os
from datetime import datetime
from google.colab import drive

# Setup
drive.mount('/content/drive', force_remount=True)
folder_path = '/content/drive/My Drive/PairTrading/data/cleansed'
out_folder_path = '/content/drive/My Drive/PairTrading/output'
if not os.path.exists(out_folder_path): os.makedirs(out_folder_path)
OUT_FILE = f'{out_folder_path}/PairTrade_Final_Metrics_{datetime.now().strftime("%Y%m%d_%H%M")}.xlsx'

def load_clean(file_name, label):
    path = os.path.join(folder_path, file_name + '.txt')
    if not os.path.exists(path): path = path.replace('.txt', '.csv')
    df = pd.read_csv(path)
    df.columns = [c.replace('<','').replace('>','').strip().upper() for c in df.columns]
    # Correcting cases where LQQ3 is measured in pounds
    if label == 'LQQ3':
        df['OPEN'] = np.where(df['OPEN'] < 500, df['OPEN'] * 100, df['OPEN'])
    return df[df['TIME'] == 160000][['DATE', 'OPEN']].rename(columns={'OPEN': label})

# Data prep
tqqq = load_clean('tqqq.us.hourly', 'TQQQ')
lqq3 = load_clean('lqq3.uk.hourly', 'LQQ3')
fx = load_clean('gbpusd.hourly', 'FX')

df = tqqq.merge(lqq3, on='DATE').merge(fx, on='DATE')
df['DATE'] = pd.to_datetime(df['DATE'], format='%Y%m%d')
df = df.set_index('DATE').sort_index()
# Converting LQQ3 to USD
df['LQQ3_USD'] = (df['LQQ3'] / 100) * df['FX']

# Excel generation
with pd.ExcelWriter(OUT_FILE, engine='xlsxwriter') as writer:
    workbook = writer.book
    f_num = workbook.add_format({'num_format': '#,##0.0000'})
    f_pct = workbook.add_format({'num_format': '0.00%'})
    f_bold = workbook.add_format({'bold': True, 'bg_color': '#D7E4BC'})

    # --- TAB 1: INPUTS ---
    ws_in = workbook.add_worksheet('Inputs')
    input_data = [
        ['EMA Window', 20],
        ['Entry Z-Score', 0.75],
        ['Exit Z-Score', 0.25],
        ['Trading Cost', 0.0015],
        ['Risk Free Rate', 0.04] # 4% for Sharpe calculation
    ]
    for r, (param, val) in enumerate(input_data):
        ws_in.write(r, 0, param)
        ws_in.write(r, 1, val)

    # --- TAB 2: ENGINE ---
    ws_calc = workbook.add_worksheet('Strategy')
    # Added Benchmark, Peak, and Drawdown columns
    headers = ['Date', 'TQQQ', 'LQQ3_USD', 'Ratio', 'EMA', 'StdDev', 'Z_Score', 'Signal', 'Return', 'Equity', 'TQQQ_Ret', 'Peak', 'Drawdown']
    for c, h in enumerate(headers): ws_calc.write(0, c, h, f_bold)

    df_res = df.reset_index()
    for i, row in df_res.iterrows():
        r = i + 2
        ws_calc.write(i+1, 0, row['DATE'].strftime('%Y-%m-%d'))
        ws_calc.write(i+1, 1, row['TQQQ'], f_num)
        ws_calc.write(i+1, 2, row['LQQ3_USD'], f_num)
        ws_calc.write_formula(i+1, 3, f"=B{r}/C{r}", f_num)

        if i == 0:
            ws_calc.write_formula(i+1, 4, f"=D{r}", f_num)
            ws_calc.write(i+1, 5, 0) # StdDev
            ws_calc.write(i+1, 7, 0) # Signal
            ws_calc.write(i+1, 9, 1.0) # Equity
            ws_calc.write(i+1, 11, 1.0) # Peak
        else:
            # Stats logic
            ws_calc.write_formula(i+1, 4, f"=(D{r}*(2/(Inputs!$B$1+1)))+(E{r-1}*(1-(2/(Inputs!$B$1+1))))", f_num)
            ws_calc.write_formula(i+1, 5, f"=STDEV.S(OFFSET(D{r}, -(MIN(Inputs!$B$1, ROW()-1)-1), 0, MIN(Inputs!$B$1, ROW()-1), 1))", f_num)
            ws_calc.write_formula(i+1, 6, f"=IF(F{r}=0, 0, (D{r}-E{r})/F{r})", f_num)

            # SIGNAL LOGIC (REVERTED TO YOUR CORRECT VERSION)
            ws_calc.write_formula(i+1, 7,
                f"=IF(H{r-1}=0, "
                f"IF(G{r}>Inputs!$B$2, -1, IF(G{r}<-Inputs!$B$2, 1, 0)), "
                f"IF(AND(H{r-1}=1, G{r}>-Inputs!$B$3), 0, "
                f"IF(AND(H{r-1}=-1, G{r}<Inputs!$B$3), 0, H{r-1})))"
            )

            # Return & Equity
            ws_calc.write_formula(i+1, 8, f"=(((B{r}/B{r-1})-(C{r}/C{r-1}))*H{r-1})-IF(H{r}<>H{r-1}, Inputs!$B$4, 0)", f_pct)
            ws_calc.write_formula(i+1, 9, f"=J{r-1}*(1+I{r})", f_num)

            # Benchmark (TQQQ) for Beta
            ws_calc.write_formula(i+1, 10, f"=(B{r}-B{r-1})/B{r-1}", f_pct)

            # Max Drawdown Tracking
            ws_calc.write_formula(i+1, 11, f"=MAX(J$2:J{r})", f_num)
            ws_calc.write_formula(i+1, 12, f"=(J{r}-L{r})/L{r}", f_pct)

    # --- TAB 3: SUMMARY ---
    ws_sum = workbook.add_worksheet('Summary')
    last_row = len(df_res) + 1

    # Advanced Metrics Formulas
    summary_metrics = [
        ['Total Strategy Return', f"='Strategy'!J{last_row}-1", f_pct],
        ['Annualized Return', f"=((1+B2)^(252/{len(df_res)}))-1", f_pct],
        ['Annualized Volatility', f"=STDEV.P('Strategy'!I2:I{last_row})*SQRT(252)", f_pct],
        ['Sharpe Ratio', f"=(B3-Inputs!B5)/B4", f_num],
        ['Max Drawdown', f"=MIN('Strategy'!M2:M{last_row})", f_pct],
        ['--- Greeks vs TQQQ ---', '', None],
        ['Beta', f"=COVARIANCE.P('Strategy'!I2:I{last_row}, 'Strategy'!K2:K{last_row})/VAR.P('Strategy'!K2:K{last_row})", f_num],
        ['Correlation', f"=CORREL('Strategy'!I2:I{last_row}, 'Strategy'!K2:K{last_row})", f_num],
        ['Alpha (Annualized)', f"=B3 - (Inputs!B5 + B8 * (((B{last_row}/B2)^(252/{len(df_res)}))-1 - Inputs!B5))", f_pct],
        ['--- Trade Stats ---', '', None],
        ['Long USD Trades', f"=COUNTIFS('Strategy'!H2:H{last_row}, 1, 'Strategy'!H1:H{last_row-1}, 0)", None],
        ['Long GBP Trades', f"=COUNTIFS('Strategy'!H2:H{last_row}, -1, 'Strategy'!H1:H{last_row-1}, 0)", None]
    ]

    ws_sum.write('A1', 'Metric', f_bold); ws_sum.write('B1', 'Value', f_bold)
    for r, (m, form, fmt) in enumerate(summary_metrics):
        ws_sum.write(r+1, 0, m)
        if form != '':
            if fmt: ws_sum.write_formula(r+1, 1, form, fmt)
            else: ws_sum.write_formula(r+1, 1, form)


print(f"✅ FINAL AUDIT WITH GREEKS COMPLETE: {OUT_FILE}")

Mounted at /content/drive
✅ FINAL AUDIT WITH GREEKS COMPLETE: /content/drive/My Drive/PairTrading/output/PairTrade_Final_Metrics_20260706_1017.xlsx
